In [ ]:
%matplotlib qt

import numpy as np
import hyperspy.api as hs
import scipy as spy
import matplotlib as plt
from hyperspy.signals import Signal2D

plt.rcParams.update({'font.size': 20})

In [ ]:
#Load data
Image = hs.load(r"C:\Users\phycbre\Downloads\1042 3.40 Mx GO3_B.png")
Image = Signal2D(Image.data)
#Normalise data if required
Image.data = Image.data.astype('float32')
Image.data *= 1 / Image.data.max()

#Rotate if necessary
Image.map(spy.ndimage.rotate, angle=0, reshape=False)

Scale = 0.0280426248 #nm/pixel
for ax in Image.axes_manager.signal_axes:
    ax.scale = Scale
    ax.units = "nm"

Image.plot()

In [ ]:
#Select the region for cropping using the widget for interactive area
reg = hs.roi.RectangularROI(left=-100, top=-10, right=300, bottom=300)
Image.plot(cmap='magma', vmax=0.8)
reg.add_widget(Image)

In [ ]:
#Fourier transform image
Image = reg(Image)
FourierTransform = Image.fft(shift = True, apodization="hamming")
FourierTransform.plot(power_spectrum = True, vmin="20th", cmap = 'inferno')

In [ ]:
#Select the region for cropping using the widget for interactive area
reg = hs.roi.RectangularROI(left=-100, top=-10, right=300, bottom=300)
FourierTransform.plot(cmap='gray', vmin="20th")
reg.add_widget(FourierTransform)

In [ ]:
PowerSpectrum = np.real(FourierTransform)*np.real(FourierTransform)+np.imag(FourierTransform)*np.imag(FourierTransform)
PowerSpectrum.data *= 1/PowerSpectrum.data.max()

PowerSpectrum.plot(vmax =0.00005,
                   norm='linear',
                  cmap = 'inferno_r')

In [ ]:
#Select the region for cropping using the widget for interactive area
reg = hs.roi.CircleROI(cx=0, cy=0, r=5, r_inner=0)
PowerSpectrum.plot(cmap='gray', vmax =0.00005)
reg.add_widget(PowerSpectrum)

In [ ]:
#Move the roi using the widget in the previous cell and then run this cell
cx = reg.cx
cy = reg.cy
r = reg.r

ax_y = PowerSpectrum.axes_manager.signal_axes[0]
ax_x = PowerSpectrum.axes_manager.signal_axes[1]

cx_pix = ax_x.value2index(reg.cx)
cy_pix = ax_y.value2index(reg.cy)

r_pix = reg.r / ax_x.scale

ny, nx = PowerSpectrum.data.shape
Y, X = np.ogrid[:ny, :nx]

Mask = (X - cx_pix)**2 + (Y - cy_pix)**2 <= r_pix*r_pix

FriedelPair = np.fliplr(np.flipud(Mask))

Mask = Mask + FriedelPair

MaskedSpectrum = PowerSpectrum.deepcopy()
MaskedSpectrum.data[~Mask] = 0
MaskedSpectrum.plot(cmap='gray', vmax =0.00005)

In [ ]:
Signal2D(Mask).plot()

In [ ]:
InverseFourierTransform = MaskedSpectrum.fft(shift = True, apodization="hamming")

InversePowerSpectrum = np.real(InverseFourierTransform)*np.real(InverseFourierTransform)+np.imag(InverseFourierTransform)*np.imag(InverseFourierTransform)
InversePowerSpectrum.data *= 1/PowerSpectrum.data.max()

InversePowerSpectrum.plot(vmin="10th", cmap = 'inferno_r')

In [ ]:
#Crop the Fourier transformed image into a region "BoxRadius" around the center
Shape = FourierTransform.data.shape
x_center = Shape[0]//2
y_center = Shape[1]//2
BoxRadius = 300

fft_cropped = FourierTransform.isig[x_center-BoxRadius:x_center+BoxRadius, y_center-BoxRadius:y_center+BoxRadius]

fft_cropped.plot(power_spectrum = True,
                 vmin="90th", 
                 cmap = 'inferno')

In [ ]:
fft_cropped = FourierTransform

In [ ]:
#Convert to an array
Data_Array = np.array(fft_cropped)

In [ ]:
#Calculate the power spectrum and apply a cutoff to clean up image
Power_Spectrum = np.log(np.real(Data_Array*np.conjugate(Data_Array)))

BoolTest = Power_Spectrum > 0.5*Power_Spectrum.max()

Power_Spectrum = BoolTest*Power_Spectrum

In [ ]:
Power_Data = Signal2D(Power_Spectrum)
Power_Data.calibrate(x0=0, y0=0, x1=50, y1=50, new_length=9, units="nm^-1", interactive=False)
Power_Data.plot(norm='linear',
               cmap = 'inferno', vmin = '20th',
               )

In [ ]:
#Find the location of all the spots in the power spectrum
A = Power_Data.find_peaks(method='minmax', interactive=False)

#Recenter coordinates
Center = Power_Data.data.shape
PeakCoords = A.data[0] - (np.array(Center)//2 +1)
    # method='zaefferer', grad_threshold=0.1, window_size=40, distance_cutoff=50.0)

In [ ]:
#Convert to polar coordinates if needed
NumberOfSpots = len(PeakCoords)
PolarCoords = np.zeros((NumberOfSpots, 2))

i = 0
while i < NumberOfSpots:
    
    PolarCoords[i][0] = np.sqrt(PeakCoords[i][0]**2 + PeakCoords[i][1]**2)
    PolarCoords[i][1] = np.arctan(PeakCoords[i][0]/PeakCoords[i][1])
    i += 1 
    
PolarCoords

In [ ]:
Power_Data.plot(norm='log',
               cmap = 'inferno',
               vmin = '60th')